In [1]:
%env CUDA_VISIBLE_DEVICES=6

env: CUDA_VISIBLE_DEVICES=6


In [2]:
import copy
import torch
from torchvision.transforms.functional import resize
import sys

sys.path.append("/home/gaoya/Code_Video/WMReward-main")
from utils import load_vjepa_model_source,compute_vjepa_loss_sliding_window,get_video
from transformers import AutoVideoProcessor, AutoModel
encoder, target_encoder, predictor, img_size = load_vjepa_model_source(
    model="vitg384",
    num_frames=64,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = encoder.to(device).eval()
target_encoder = target_encoder.to(device).eval()
predictor = predictor.to(device).eval()



# ckpt_path = "/data/gaoya/ckpt/facebook-vjepa2-vitg-fpc64-384/"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = AutoModel.from_pretrained(ckpt_path, local_files_only=True).to(device).eval()
# processor = AutoVideoProcessor.from_pretrained(ckpt_path)


/data/gaoya/miniconda3/envs/vjepa2/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
from decord import VideoReader
video_path = "/home/gaoya/Code_Video/vjepa2-main/assets/holding_phone.mp4"
vr = VideoReader(video_path)
import numpy as np
import torch
from decord import VideoReader
from torchvision.transforms.functional import resize


In [17]:
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import resize
import matplotlib.pyplot as plt

from utils_my import (
    get_video,
    build_pt_video_transform,
    generate_vjepa_masks,
    apply_masks,
)

# --------------------------------------------------
# 1) 保留你原来的视频读取函数
# --------------------------------------------------


import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from torchvision.transforms.functional import resize


def load_video_as_tensor(video_path, start_frame=0, end_frame=None, num_samples=64, img_size=384):
    vr = VideoReader(video_path)
    total_frames = len(vr)
    print(f"total_frames = {total_frames}")

    if end_frame is None:
        end_frame = total_frames

    start_frame = max(0, start_frame)
    end_frame = min(end_frame, total_frames)
    assert start_frame < end_frame, f"start_frame({start_frame}) must < end_frame({end_frame})"

    num_samples = min(num_samples, end_frame - start_frame)
    frame_idx = np.linspace(start_frame, end_frame - 1, num_samples, dtype=int)

    video_np = vr.get_batch(frame_idx).asnumpy()

    video_tensor = torch.from_numpy(video_np).permute(3, 0, 1, 2).float()
    video_tensor = resize(video_tensor.permute(1, 0, 2, 3), [img_size, img_size])
    video_tensor = video_tensor.permute(1, 0, 2, 3)
    video_tensor = (video_tensor / 127.5) - 1.0

    return video_tensor.unsqueeze(0), frame_idx, start_frame, end_frame



def build_prefix_to_last_masks(batch_size, total_frames, context_frames, pred_frames, img_size, encoder, device):
    patch_size = encoder.patch_size
    tubelet_size = encoder.tubelet_size

    assert context_frames % tubelet_size == 0, \
        f"context_frames={context_frames} 必须能被 tubelet_size={tubelet_size} 整除"
    assert pred_frames % tubelet_size == 0, \
        f"pred_frames={pred_frames} 必须能被 tubelet_size={tubelet_size} 整除"
    assert context_frames + pred_frames <= total_frames, \
        f"context_frames + pred_frames = {context_frames + pred_frames} 不能超过 total_frames={total_frames}"

    grid_size = img_size // patch_size
    tokens_per_step = grid_size * grid_size
    time_steps = total_frames // tubelet_size

    context_steps = context_frames // tubelet_size
    pred_steps = pred_frames // tubelet_size

    enc_end = context_steps * tokens_per_step
    pred_start = (time_steps - pred_steps) * tokens_per_step
    pred_end = time_steps * tokens_per_step

    masks_enc = torch.arange(0, enc_end, device=device).unsqueeze(0).repeat(batch_size, 1)
    masks_pred = torch.arange(pred_start, pred_end, device=device).unsqueeze(0).repeat(batch_size, 1)

    meta = {
        "patch_size": patch_size,
        "tubelet_size": tubelet_size,
        "grid_size": grid_size,
        "tokens_per_step": tokens_per_step,
        "time_steps": time_steps,
        "context_steps": context_steps,
        "pred_steps": pred_steps,
        "enc_token_range": (0, enc_end),
        "pred_token_range": (pred_start, pred_end),
    }
    return masks_enc, masks_pred, meta


@torch.no_grad()
def predict_video_features(
    video_tensor,              # [B,C,T,H,W] or [C,T,H,W], range [-1,1]
    encoder,
    target_encoder,
    predictor,
    img_size,
    context_frames=8,
    pred_frames=8,
    mode="next",              # "next" or "last"
    loss_type="mse",          # "mse" / "l1" / "cosine"
):
    device = next(encoder.parameters()).device
    model_dtype = next(encoder.parameters()).dtype

    if video_tensor.dim() == 4:
        video_tensor = video_tensor.unsqueeze(0)

    video_tensor = video_tensor.to(device=device, dtype=model_dtype)
    B, C, T, H, W = video_tensor.shape
    assert T % encoder.tubelet_size == 0, \
        f"T={T} 必须能被 tubelet_size={encoder.tubelet_size} 整除"
    if mode == "next":
        frames_per_clip = context_frames + pred_frames
        assert T >= frames_per_clip, f"视频总帧数 {T} 小于 context+pred={frames_per_clip}"
        clip = video_tensor[:, :, :frames_per_clip]

    elif mode == "last":
        clip = video_tensor
        frames_per_clip = T
        assert context_frames + pred_frames <= T, \
            f"context_frames + pred_frames = {context_frames + pred_frames} 不能超过总帧数 {T}"
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # 统一复用你原来的预处理逻辑
    transform = build_pt_video_transform(img_size)
    video_255 = (clip + 1.0) * 127.5

    batch_processed = []
    for b in range(B):
        video_tcthw = video_255[b].permute(1, 0, 2, 3)   # [T,C,H,W]
        video_normalized = transform(video_tcthw)        # [C,T,H,W]
        batch_processed.append(video_normalized)

    clip = torch.stack(batch_processed, dim=0).to(model_dtype)  # [B,C,T,H,W]

    # 构造 mask
    if mode == "next":
        masks_enc, masks_pred = generate_vjepa_masks(
            masking_mode="causal",
            batch_size=B,
            img_size=img_size,
            frames_per_clip=frames_per_clip,
            encoder=encoder,
            context_frames=context_frames,
            device=device,
        )
        mask_meta = {
            "patch_size": encoder.patch_size,
            "tubelet_size": encoder.tubelet_size,
            "grid_size": img_size // encoder.patch_size,
        }
    else:
        masks_enc, masks_pred, mask_meta = build_prefix_to_last_masks(
            batch_size=B,
            total_frames=T,
            context_frames=context_frames,
            pred_frames=pred_frames,
            img_size=img_size,
            encoder=encoder,
            device=device,
        )

    # GT future / last
    h = target_encoder(clip)
    h = torch.stack([F.layer_norm(hi, (hi.size(-1),)) for hi in h])
    gt = apply_masks(h, masks_pred, concat=False)[0]   # [B, N_pred, D]

    # Pred future / last
    z = encoder(clip, masks_enc)
    z = predictor(z, masks_enc, masks_pred)
    pred = F.layer_norm(z, (z.size(-1),))

    # loss
    mse = F.mse_loss(pred, gt, reduction="mean")
    l1 = F.l1_loss(pred, gt, reduction="mean")
    cosine_loss = 1.0 - F.cosine_similarity(pred, gt, dim=-1).mean()

    if loss_type == "mse":
        loss = mse
    elif loss_type == "l1":
        loss = l1
    elif loss_type == "cosine":
        loss = cosine_loss
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    return {
        "pred": pred,
        "gt": gt,
        "loss": loss,
        "mse": mse,
        "l1": l1,
        "cosine_loss": cosine_loss,
        "meta": {
            "mode": mode,
            "video_shape": tuple(video_tensor.shape),
            "clip_shape": tuple(clip.shape),
            "pred_shape": tuple(pred.shape),
            "gt_shape": tuple(gt.shape),
            "context_frames": context_frames,
            "pred_frames": pred_frames,
            "num_context_tokens": masks_enc.shape[1],
            "num_pred_tokens": masks_pred.shape[1],
            **mask_meta,
        },
        "masks_enc": masks_enc,
        "masks_pred": masks_pred,
    }



def sweep_context_for_last8_prediction(
    video_path,
    encoder,
    target_encoder,
    predictor,
    img_size,
    context_list=(4, 8, 12, 16, 20, 24, 32, 40),
    pred_frames=8,
    start_frame=0,
    end_frame=None,
    num_samples=64,
):
    video_tensor, frame_idx, real_start_frame, real_end_frame = load_video_as_tensor(
        video_path=video_path,
        start_frame=start_frame,
        end_frame=end_frame,
        num_samples=num_samples,
        img_size=img_size,
    )
    print(f"sample_range = [{real_start_frame}, {real_end_frame})")
    T = video_tensor.shape[2]
    tubelet_size = encoder.tubelet_size

    results = []
    print("=" * 80)
    print(f"video_path = {video_path}")
    print(f"sample_range = [{start_frame}, {end_frame})")
    print(f"sampled_frames = {T}, pred_frames = {pred_frames}")
    print(f"sampled frame_idx = {frame_idx.tolist()}")
    print("=" * 80)

    for context_frames in context_list:
        if context_frames + pred_frames > T:
            print(f"[skip] context={context_frames}: context + pred > sampled total_frames")
            continue
        if context_frames % tubelet_size != 0:
            print(f"[skip] context={context_frames}: not divisible by tubelet_size={tubelet_size}")
            continue
        if pred_frames % tubelet_size != 0:
            raise ValueError(f"pred_frames={pred_frames} must be divisible by tubelet_size={tubelet_size}")

        out = predict_video_features(
            video_tensor=video_tensor,
            encoder=encoder,
            target_encoder=target_encoder,
            predictor=predictor,
            img_size=img_size,
            context_frames=context_frames,
            pred_frames=pred_frames,
            mode="last",
            loss_type="mse",
        )

        row = {
            "context_frames": context_frames,
            "mse": out["mse"].item(),
            "l1": out["l1"].item(),
            "cosine_loss": out["cosine_loss"].item(),
            "pred_shape": out["meta"]["pred_shape"],
            "gt_shape": out["meta"]["gt_shape"],
            "num_context_tokens": out["meta"]["num_context_tokens"],
            "num_pred_tokens": out["meta"]["num_pred_tokens"],
            "sampled_frame_idx": frame_idx.tolist(),
            "context_frame_idx": frame_idx[:context_frames].tolist(),
            "pred_frame_idx": frame_idx[-pred_frames:].tolist(),
        }
        results.append(row)

        print(
            f"context={context_frames:>2d} | "
            f"MSE={row['mse']:.6f} | "
            f"L1={row['l1']:.6f} | "
            f"CosLoss={row['cosine_loss']:.6f} | "
            f"ctx_tokens={row['num_context_tokens']} | "
            f"pred_tokens={row['num_pred_tokens']} | "
            f"pred_frame_idx={row['pred_frame_idx']}"
        )

    print("=" * 80)
    return results

In [20]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from decord import VideoReader
from torchvision.transforms.functional import resize

from utils_my import (
    build_pt_video_transform,
    generate_vjepa_masks,
    apply_masks,
)


# --------------------------------------------------
# 1) 视频读取：支持指定原视频区间并均匀采样
#    返回：
#      video_tensor: [1, C, T, H, W], range [-1, 1]
#      frame_idx:    采样到的原视频帧索引
#      real_start_frame / real_end_frame: 实际生效区间
# --------------------------------------------------
def load_video_as_tensor(
    video_path,
    start_frame=0,
    end_frame=None,
    num_samples=64,
    img_size=384,
):
    vr = VideoReader(video_path)
    total_frames = len(vr)
    print(f"total_frames = {total_frames}")

    if end_frame is None:
        end_frame = total_frames

    # 边界保护
    start_frame = max(0, start_frame)
    end_frame = min(end_frame, total_frames)
    assert start_frame < end_frame, \
        f"start_frame({start_frame}) must < end_frame({end_frame})"

    num_samples = min(num_samples, end_frame - start_frame)
    if num_samples <= 0:
        raise ValueError(
            f"num_samples={num_samples} 非法，"
            f"请检查 start_frame={start_frame}, end_frame={end_frame}"
        )

    # 在 [start_frame, end_frame) 内均匀采样 num_samples 帧
    frame_idx = np.linspace(start_frame, end_frame - 1, num_samples, dtype=int)

    # 直接按索引取帧: [T, H, W, C], uint8
    video_np = vr.get_batch(frame_idx).asnumpy()

    # -> [C, T, H, W]
    video_tensor = torch.from_numpy(video_np).permute(3, 0, 1, 2).float()

    # resize 需要 [T, C, H, W]
    video_tensor = resize(
        video_tensor.permute(1, 0, 2, 3),
        [img_size, img_size]
    )
    # 再转回 [C, T, H, W]
    video_tensor = video_tensor.permute(1, 0, 2, 3)

    # [0,255] -> [-1,1]
    video_tensor = (video_tensor / 127.5) - 1.0

    return video_tensor.unsqueeze(0), frame_idx, start_frame, end_frame


# --------------------------------------------------
# 2) 构造“前缀 -> 最后 pred_frames”的 token mask
# --------------------------------------------------
def build_prefix_to_last_masks(
    batch_size,
    total_frames,
    context_frames,
    pred_frames,
    img_size,
    encoder,
    device,
):
    patch_size = encoder.patch_size
    tubelet_size = encoder.tubelet_size

    assert context_frames % tubelet_size == 0, \
        f"context_frames={context_frames} 必须能被 tubelet_size={tubelet_size} 整除"
    assert pred_frames % tubelet_size == 0, \
        f"pred_frames={pred_frames} 必须能被 tubelet_size={tubelet_size} 整除"
    assert context_frames + pred_frames <= total_frames, \
        f"context_frames + pred_frames = {context_frames + pred_frames} 不能超过 total_frames={total_frames}"

    grid_size = img_size // patch_size
    tokens_per_step = grid_size * grid_size
    time_steps = total_frames // tubelet_size

    context_steps = context_frames // tubelet_size
    pred_steps = pred_frames // tubelet_size

    enc_end = context_steps * tokens_per_step
    pred_start = (time_steps - pred_steps) * tokens_per_step
    pred_end = time_steps * tokens_per_step

    masks_enc = torch.arange(0, enc_end, device=device).unsqueeze(0).repeat(batch_size, 1)
    masks_pred = torch.arange(pred_start, pred_end, device=device).unsqueeze(0).repeat(batch_size, 1)

    meta = {
        "patch_size": patch_size,
        "tubelet_size": tubelet_size,
        "grid_size": grid_size,
        "tokens_per_step": tokens_per_step,
        "time_steps": time_steps,
        "context_steps": context_steps,
        "pred_steps": pred_steps,
        "enc_token_range": (0, enc_end),
        "pred_token_range": (pred_start, pred_end),
    }
    return masks_enc, masks_pred, meta


# --------------------------------------------------
# 3) 通用特征预测函数
#    mode="next": 前 context 预测紧邻后 pred
#    mode="last": 前 context 预测整段视频最后 pred
# --------------------------------------------------
@torch.no_grad()
def predict_video_features(
    video_tensor,              # [B,C,T,H,W] or [C,T,H,W], range [-1,1]
    encoder,
    target_encoder,
    predictor,
    img_size,
    context_frames=8,
    pred_frames=8,
    mode="next",              # "next" or "last"
    loss_type="mse",          # "mse" / "l1" / "cosine"
):
    device = next(encoder.parameters()).device
    model_dtype = next(encoder.parameters()).dtype

    if video_tensor.dim() == 4:
        video_tensor = video_tensor.unsqueeze(0)

    video_tensor = video_tensor.to(device=device, dtype=model_dtype)
    B, C, T, H, W = video_tensor.shape

    assert T % encoder.tubelet_size == 0, \
        f"T={T} 必须能被 tubelet_size={encoder.tubelet_size} 整除"

    if mode == "next":
        frames_per_clip = context_frames + pred_frames
        assert T >= frames_per_clip, \
            f"视频总帧数 {T} 小于 context+pred={frames_per_clip}"
        clip = video_tensor[:, :, :frames_per_clip]

    elif mode == "last":
        clip = video_tensor
        frames_per_clip = T
        assert context_frames + pred_frames <= T, \
            f"context_frames + pred_frames = {context_frames + pred_frames} 不能超过总帧数 {T}"
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # 复用现有预处理
    transform = build_pt_video_transform(img_size)
    video_255 = (clip + 1.0) * 127.5

    batch_processed = []
    for b in range(B):
        video_tcthw = video_255[b].permute(1, 0, 2, 3)   # [T,C,H,W]
        video_normalized = transform(video_tcthw)        # [C,T,H,W]
        batch_processed.append(video_normalized)

    clip = torch.stack(batch_processed, dim=0).to(model_dtype)  # [B,C,T,H,W]

    # 构造 mask
    if mode == "next":
        masks_enc, masks_pred = generate_vjepa_masks(
            masking_mode="causal",
            batch_size=B,
            img_size=img_size,
            frames_per_clip=frames_per_clip,
            encoder=encoder,
            context_frames=context_frames,
            device=device,
        )
        mask_meta = {
            "patch_size": encoder.patch_size,
            "tubelet_size": encoder.tubelet_size,
            "grid_size": img_size // encoder.patch_size,
        }
    else:
        masks_enc, masks_pred, mask_meta = build_prefix_to_last_masks(
            batch_size=B,
            total_frames=T,
            context_frames=context_frames,
            pred_frames=pred_frames,
            img_size=img_size,
            encoder=encoder,
            device=device,
        )

    # GT
    h = target_encoder(clip)
    h = torch.stack([F.layer_norm(hi, (hi.size(-1),)) for hi in h])
    gt = apply_masks(h, masks_pred, concat=False)[0]   # [B, N_pred, D]

    # Pred
    z = encoder(clip, masks_enc)
    z = predictor(z, masks_enc, masks_pred)
    pred = F.layer_norm(z, (z.size(-1),))

    # loss
    mse = F.mse_loss(pred, gt, reduction="mean")
    l1 = F.l1_loss(pred, gt, reduction="mean")
    cosine_loss = 1.0 - F.cosine_similarity(pred, gt, dim=-1).mean()

    if loss_type == "mse":
        loss = mse
    elif loss_type == "l1":
        loss = l1
    elif loss_type == "cosine":
        loss = cosine_loss
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    return {
        "pred": pred,
        "gt": gt,
        "loss": loss,
        "mse": mse,
        "l1": l1,
        "cosine_loss": cosine_loss,
        "meta": {
            "mode": mode,
            "video_shape": tuple(video_tensor.shape),
            "clip_shape": tuple(clip.shape),
            "pred_shape": tuple(pred.shape),
            "gt_shape": tuple(gt.shape),
            "context_frames": context_frames,
            "pred_frames": pred_frames,
            "num_context_tokens": masks_enc.shape[1],
            "num_pred_tokens": masks_pred.shape[1],
            **mask_meta,
        },
        "masks_enc": masks_enc,
        "masks_pred": masks_pred,
    }


# --------------------------------------------------
# 4) sweep：预测采样序列最后 pred_frames 帧
# --------------------------------------------------
def sweep_context_for_last8_prediction(
    video_path,
    encoder,
    target_encoder,
    predictor,
    img_size,
    context_list=(4, 8, 12, 16, 20, 24, 32, 40),
    pred_frames=8,
    start_frame=0,
    end_frame=None,
    num_samples=64,
):
    video_tensor, frame_idx, real_start_frame, real_end_frame = load_video_as_tensor(
        video_path=video_path,
        start_frame=start_frame,
        end_frame=end_frame,
        num_samples=num_samples,
        img_size=img_size,
    )

    T = video_tensor.shape[2]
    tubelet_size = encoder.tubelet_size

    results = []
    print("=" * 80)
    print(f"video_path = {video_path}")
    print(f"sample_range = [{real_start_frame}, {real_end_frame})")
    print(f"sampled_frames = {T}, pred_frames = {pred_frames}")
    print(f"sampled frame_idx = {frame_idx.tolist()}")
    print("=" * 80)

    for context_frames in context_list:
        if context_frames + pred_frames > T:
            print(f"[skip] context={context_frames}: context + pred > sampled total_frames")
            continue
        if context_frames % tubelet_size != 0:
            print(f"[skip] context={context_frames}: not divisible by tubelet_size={tubelet_size}")
            continue
        if pred_frames % tubelet_size != 0:
            raise ValueError(
                f"pred_frames={pred_frames} must be divisible by tubelet_size={tubelet_size}"
            )

        out = predict_video_features(
            video_tensor=video_tensor,
            encoder=encoder,
            target_encoder=target_encoder,
            predictor=predictor,
            img_size=img_size,
            context_frames=context_frames,
            pred_frames=pred_frames,
            mode="last",
            loss_type="mse",
        )

        row = {
            "context_frames": context_frames,
            "mse": out["mse"].item(),
            "l1": out["l1"].item(),
            "cosine_loss": out["cosine_loss"].item(),
            "pred_shape": out["meta"]["pred_shape"],
            "gt_shape": out["meta"]["gt_shape"],
            "num_context_tokens": out["meta"]["num_context_tokens"],
            "num_pred_tokens": out["meta"]["num_pred_tokens"],
            "sampled_frame_idx": frame_idx.tolist(),
            "context_frame_idx": frame_idx[:context_frames].tolist(),
            "pred_frame_idx": frame_idx[-pred_frames:].tolist(),
        }
        results.append(row)

        print(
            f"context={context_frames:>2d} | "
            f"MSE={row['mse']:.6f} | "
            f"L1={row['l1']:.6f} | "
            f"CosLoss={row['cosine_loss']:.6f} | "
            f"ctx_tokens={row['num_context_tokens']} | "
            f"pred_tokens={row['num_pred_tokens']} | "
            f"pred_frame_idx={row['pred_frame_idx']}"
        )

    print("=" * 80)
    return results


total_frames = 60
torch.Size([1, 3, 48, 384, 384])
[ 0  1  2  3  5  6  7  8 10 11 12 13 15 16 17 18 20 21 22 23 25 26 27 28
 30 31 32 33 35 36 37 38 40 41 42 43 45 46 47 48 50 51 52 53 55 56 57 59]
0 60
0.8156996965408325
{'mode': 'next', 'video_shape': (1, 3, 48, 384, 384), 'clip_shape': (1, 3, 48, 384, 384), 'pred_shape': (1, 2304, 1408), 'gt_shape': (1, 2304, 1408), 'context_frames': 40, 'pred_frames': 8, 'num_context_tokens': 11520, 'num_pred_tokens': 2304, 'patch_size': 16, 'tubelet_size': 2, 'grid_size': 24}
total_frames = 60
video_path = /data/gaoya/AAA_test_video/Dataset_test/genesis_sim_v5/train/scene_000001/video/preview.mp4
sample_range = [0, 60)
sampled_frames = 60, pred_frames = 8
sampled frame_idx = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
context= 4 | MSE=0.973982 | L1=0.602163 | CosL

In [21]:
print("/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/finaljson/712.json")

/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/finaljson/712.json


In [ ]:


# ==================================================
# 5) 示例调用
# ==================================================
video_path = "/data/gaoya/AAA_test_video/Dataset_test/genesis_sim_v5/train/scene_000001/video/preview.mp4"
img_size = 384

# 注意：这里现在要接 4 个返回值
video_tensor, frame_idx, real_start_frame, real_end_frame = load_video_as_tensor(
    video_path=video_path,
    start_frame=0,
    end_frame=None,
    num_samples=48,   # 建议取能被 tubelet_size 整除的值
    img_size=img_size,
)

print(video_tensor.shape)   # [1, C, T, H, W]
print(frame_idx)
print(real_start_frame, real_end_frame)

# 单次测试：前40帧预测紧邻后8帧
out = predict_video_features(
    video_tensor=video_tensor,
    encoder=encoder,
    target_encoder=target_encoder,
    predictor=predictor,
    img_size=img_size,
    context_frames=40,
    pred_frames=8,
    mode="next",
    loss_type="mse",
)

print(out["loss"].item())
print(out["meta"])

# sweep：前缀预测采样序列最后8帧
results = sweep_context_for_last8_prediction(
    video_path=video_path,
    encoder=encoder,
    target_encoder=target_encoder,
    predictor=predictor,
    img_size=img_size,
    context_list=(4, 8, 12, 16, 20, 24, 32, 40),
    pred_frames=8,
    start_frame=0,
    end_frame=640,
    num_samples=64,   # 这里也建议保证能被 tubelet_size 整除
)

print("\nSummary:")
for r in results:
    print(r)